<a href="https://colab.research.google.com/github/Arda-Avci/AI-Publisher/blob/main/Google_Colab_AI_Publisher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **ESKI MIMARI** — Bu notebook Docker Supervisor mimarisine gecilmeden onceki surumdur.
> **colab_setup.ipynb** kullanin.
> Geriye donuk uyumluluk icin korunuyor.


# AI Publisher - Otonom Video Uretim ve Pazarlama Sunucusu (LEGACY)

Bu notebook, AI Publisher projesi icin Google Colab uzerinde calisan GPU destekli yapay zeka sunucusunu (CogVideoX-5b, XTTS-v2, AudioLDM2, GFPGAN/RealESRGAN ve Wav2Lip dudak senkronizasyonu) kurar ve baslatir.

### Kurulum ve Baslatma Talimati:
1. Ust menuden **Duzenle (Edit) > Defter Ayarlari (Notebook Settings)** secenegine gidin.
2. Donanim ivmelendiriciyi **T4 GPU** (veya daha ustu bir ekran karti) olarak secip kaydedin.
3. Asagidaki kod hucresini calistirin. Bagimliliklar ilk kez kurulurken kernel otomatik olarak yeniden baslatilacaktir (Oturum coktu uyarisi normaldir).
4. Kernel otomatik kapandiktan sonra **ayni hucreyi ikinci kez calistirin**.
5. Ngrok Auth Token istendiginde ngrok.com adresinden alacaginiz token'inizi girin.
6. Sunucu basariyla ayaga kalktiginda size verilen tunel URL'sini kopyalayip yerel Node.js projenizin `.env` dosyasindaki `COLAB_URL` alanina yapistirin.

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
from google.colab import drive
drive.mount('/content/drive')

GITHUB_TOKEN = "" #@param {type:"string"}
NGROK_TOKEN = "" #@param {type:"string"}

import os, subprocess, shutil, sys
from google.colab import userdata

repo_dir = "/content/AI-Publisher"

if os.path.exists(repo_dir):
    print("Deleting existing repository directory...")
    shutil.rmtree(repo_dir, ignore_errors=True)

if not GITHUB_TOKEN:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    GITHUB_TOKEN = input("GitHub token: ").strip()

if not NGROK_TOKEN:
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
if not NGROK_TOKEN:
    NGROK_TOKEN = input("Ngrok Auth Token: ").strip()

repo_url = f"https://{GITHUB_TOKEN}@github.com/Arda-Avci/AI-Publisher.git"

print("Repository klonlaniyor...")
subprocess.run(["git", "clone", repo_url, repo_dir], check=True, env=_clean_env)

print("colab_setup.py kopyalaniyor...")
shutil.copy(os.path.join(repo_dir, "colab_setup.py"), "/content/colab_setup.py")
print("colab_server.py kopyalaniyor...")
shutil.copy(os.path.join(repo_dir, "colab_server.py"), "/content/colab_server.py")

print("Kurulum betigi canli loglar esliginde calistiriliyor...")
try:
    env = os.environ.copy()
    env["GITHUB_TOKEN"] = GITHUB_TOKEN
    env["NGROK_TOKEN"] = NGROK_TOKEN
    
    process = subprocess.Popen(
        ["python3", "-u", "/content/colab_setup.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    , env=_clean_env)
    # Canli loglari yazdir
    for line in process.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
    process.wait()
    
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, process.args)
except subprocess.CalledProcessError as e:
    if e.returncode in [9, -9, 137, 100]:
        print(f"\n[INFO] Kurulum betigi sonlandi (Cikis Kodu: {e.returncode}). Oturum yenileniyor...")
        import os
        os.kill(os.getpid(), 9)
    else:
        print(f"\nKurulum hatayla bitti! Cikis Kodu: {e.returncode}")
        raise e

## Secenek C: Tum Docker Imajlarini Insa Et ve Google Drive'a Kaydet (Maliyet Tasarruflu CPU Modu)

Bu bolum, AI Publisher projesinde kullanilan 11 Docker konteyner imajini Google Colab CPU calisma zamaninda sifirdan insa eder, pigz kullanarak cok cekirdekli paralel sikistirir, Google Drive'a yukler ve butunluk kontroluyle dogrular.

### Maliyet Tasarrufu ve Calistirma Talimati:
1. Ust menuden **Duzenle (Edit) > Defter Ayarlari (Notebook Settings)** secenegine gidin.
2. Donanim ivmelendiriciyi **CPU** (GPU yok) olarak secip kaydedin (CPU modu ucretsizdir veya cok daha ucuzdur).
3. Asagidaki hucreyi calistirin. Hucre, tum imajlari sirayla derleyecek, Drive'a .tar.gz olarak kaydedecek ve butunluklerini dogrulayacaktir.
4. **Otomatik Kapanma (Unassign):** Derleme ve dogrulama tamamlandiginda, AUTO_DISCONNECT = True olarak ayarlandiysa, Colab oturumu otomatik olarak sonlandirilacaktir. Bu sayede islem bittiginde ek sure icin ucretlendirilmezsiniz.

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Secenek C: Tum Docker Imajlarini Insa Et (Maliyet Tasarruflu CPU Modu)
GITHUB_TOKEN = "" #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

import subprocess
import os
import time
import urllib.request
import sys
import shutil
from google.colab import userdata

# 1. Depoyu Klonla (Eger yoksa)
repo_dir = "/content/AI-Publisher"
if not os.path.exists(repo_dir):
  print("Repository klonlaniyor...")
  if not GITHUB_TOKEN:
    try:
      GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except: pass
  if not GITHUB_TOKEN:
    GITHUB_TOKEN = input("GitHub token: ").strip()
  
  repo_url = f"https://{GITHUB_TOKEN}@github.com/Arda-Avci/AI-Publisher.git"
  subprocess.run(["git", "clone", repo_url, repo_dir], check=True, env=_clean_env)

# 2. Docker Kurulumu ve Daemon Baslatilmasi
if not shutil.which("docker"):
  print("[INFO] Docker.io paketi kuruluyor...")
  subprocess.run("apt-get update -q && apt-get install -y -q docker.io", shell=True, check=True, env=_clean_env)

res = subprocess.run(["docker", "info"], capture_output=True, env=_clean_env)
if res.returncode != 0:
  print("[INFO] Docker Daemon aktif degil, arkada baslatiliyor...")
  subprocess.run("pkill -9 dockerd || true", shell=True, env=_clean_env)
  subprocess.Popen(["dockerd", "-b", "none", "--iptables=0", "--storage-driver=vfs"], stdout=open("dockerd.log", "w"), stderr=subprocess.STDOUT, env=_clean_env)
  for _ in range(15):
    res = subprocess.run(["docker", "info"], capture_output=True, env=_clean_env)
    if res.returncode == 0:
      print("[OK] Docker basariyla baslatildi!")
      break
    time.sleep(1)

# 3. Go Registry Indir ve Kur
if not os.path.exists("/usr/local/bin/registry"):
  print("[INFO] Docker Registry binary indiriliyor...")
  subprocess.run("wget -q https://github.com/distribution/distribution/releases/download/v2.8.2/registry_2.8.2_linux_amd64.tar.gz", shell=True, env=_clean_env)
  subprocess.run("tar -xzf registry_2.8.2_linux_amd64.tar.gz registry", shell=True, env=_clean_env)
  subprocess.run("mv registry /usr/local/bin/registry", shell=True, env=_clean_env)
  subprocess.run("rm -f registry_2.8.2_linux_amd64.tar.gz", shell=True, env=_clean_env)
  subprocess.run("chmod +x /usr/local/bin/registry", shell=True, env=_clean_env)

# 4. Kaniko Indir ve Kur (Docker Imajindan Kopyalayarak)
if not os.path.exists("/usr/local/bin/kaniko"):
  print("[INFO] Kaniko executor binary Docker imajindan kopyalaniyor...")
  subprocess.run("docker pull gcr.io/kaniko-project/executor:latest", shell=True, check=True, env=_clean_env)
  subprocess.run("docker create --name kaniko-temp gcr.io/kaniko-project/executor:latest", shell=True, check=True, env=_clean_env)
  subprocess.run("docker cp kaniko-temp:/kaniko/executor /usr/local/bin/kaniko", shell=True, check=True, env=_clean_env)
  subprocess.run("docker rm kaniko-temp", shell=True, check=True, env=_clean_env)
  subprocess.run("chmod +x /usr/local/bin/kaniko", shell=True, check=True, env=_clean_env)

# 5. pigz Indir (paralel gzip sikistirma icin)
subprocess.run("apt-get update && apt-get install -y pigz", shell=True, env=_clean_env)

print("Kurulumlar tamamlandi.")

# 5. Registryyi Arka Planda Baslat
print("==========================================")
print("Yerel Registry localhost:5000 Portunda Baslatiliyor...")
print("==========================================")
os.makedirs("/etc/docker/registry", exist_ok=True)
with open("/etc/docker/registry/config.yml", "w") as cfg_f:
  cfg_f.write("""version: 0.1\nlog:\n  fields:\n    service: registry\nstorage:\n  cache:\n    blobdescriptor: inmemory\n  filesystem:\n    rootdirectory: /var/lib/registry\nhttp:\n  addr: :5000\n  headers:\n    X-Content-Type-Options: [nosniff]\n""")
subprocess.run("pkill -f 'registry serve' || true", shell=True, env=_clean_env)
# Arkada registry serve baslat
subprocess.Popen(["/usr/local/bin/registry", "serve", "/etc/docker/registry/config.yml"], stdout=open("registry.log", "w"), stderr=subprocess.STDOUT, env=_clean_env)

# Registry'nin hazir olmasini bekle
registry_ok = False
for _ in range(15):
  try:
    urllib.request.urlopen("http://localhost:5000/v2/")
    print("Registry basariyla baslatildi ve yanit veriyor!")
    registry_ok = True
    break
  except Exception:
    time.sleep(1)
if not registry_ok:
  print("Hata: Yerel registry baslatilamadi!")
  if os.path.exists("registry.log"):
    with open("registry.log", "r") as log_f:
      print(log_f.read())
  sys.exit(1)

# 6. Insa Betigini Tetikle
print("==========================================")
print("Docker Imajlarinin Insa Edilmesi Baslatiliyor...")
print("==========================================")

os.chdir(repo_dir)

# Son demleri git pull ile cekelim
subprocess.run("git fetch origin && git reset --hard origin/main", shell=True, env=_clean_env)
subprocess.run("chmod +x colab_docker/build_all.sh", shell=True, env=_clean_env)

# colab_docker dizinine gir
os.chdir(os.path.join(repo_dir, "colab_docker"))

# build_all.sh betigini calistir ve anlik ciktiyi al
process = subprocess.Popen(["bash", "build_all.sh"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=_clean_env)
while True:
  output = process.stdout.readline()
  if output == '' and process.poll() is not None:
    break
  if output:
    print(output.strip())

rc = process.poll()
if rc == 0:
  print("Tum imaj insalari ve Google Drive yedeklemeleri basariyla tamamlandi.")
else:
  print(f"Insa sirasinda hata olustu. Cikis kodu: {rc}")
  sys.exit(rc)
